# Конспект. Модуль 8: CatBoost изнутри

## 1. Зачем это нужно и как это связано с предыдущими модулями

Модуль 7 закрыл главный пробел вашей подготовки — LightGBM. Этот модуль закрывает **второй** явно обозначенный пробел из контекста: работу с категориальными признаками и специфику CatBoost (Яндекс, 2017). Если LightGBM (Модуль 7) оптимизирует общий фундамент Модуля 6 в первую очередь **под скорость** (leaf-wise рост, GOSS, EFB — все три нацелены на то, чтобы быстрее и дешевле обучаться на огромных данных), то CatBoost оптимизирует его **под два других приоритета**: (а) нативная, статистически корректная работа с категориальными признаками без ручного кодирования, и (б) структурная устойчивость к переобучению за счёт принципиально иного подхода к построению деревьев и организации самого процесса обучения.

Разберём два ключевых нововведения CatBoost: **Oblivious Trees (симметричные деревья)** и **Ordered Boosting**, а также то, как они вместе решают проблему категориальных признаков через **Ordered Target Statistics**.

## 2. Oblivious Trees (симметричные деревья)

### 2.1. Определение

В обычном дереве (CART, Модуль 1; leaf-wise LightGBM, Модуль 7) **каждый узел** независимо выбирает свою собственную лучшую пару (признак, порог), исходя только из тех данных, что попали именно в этот узел. **Симметричное (oblivious) дерево** устроено иначе: **на каждом уровне** дерева используется **одно и то же** правило разбиения (одна и та же пара «признак, порог») **для всех узлов этого уровня одновременно** — независимо от того, что именно попало в конкретный узел.

**Прямое следствие такой конструкции:** дерево глубины `d` **всегда** имеет ровно `2^d` листьев (дерево всегда полностью сбалансировано по построению — асимметрия, которую мы видели в LightGBM в Модуле 7, здесь структурно невозможна), и полностью описывается **всего `d` парами** (признак, порог) — по одной на уровень, а не потенциально `2^d - 1` независимыми решениями, как в обычном бинарном дереве той же глубины.

### 2.2. Как выбирается общее правило уровня — численный пример

Представим, что после разбиения корня у нас есть два узла — `L` (левый) и `R` (правый), и мы решаем, каким правилом разбить **оба** на следующем уровне. У нас есть два кандидата — `α` (признак `X2 ≤ 5`) и `β` (признак `X3 ≤ 10`), и для каждого посчитан Gain (по формуле Модуля 6) **отдельно** для `L` и **отдельно** для `R`:

| Кандидат правила | Gain при применении к L | Gain при применении к R | **Суммарный Gain** |
|---|---|---|---|
| `α: X2 ≤ 5` | 20 | 15 | **35** |
| `β: X3 ≤ 10` | 30 | 2 | **32** |

**В обычном (не симметричном) дереве** каждый узел выбрал бы своё **локально** лучшее правило независимо: `L` выбрал бы `β` (Gain=30 у него лучше, чем 20 у `α`), `R` выбрал бы `α` (Gain=15 лучше, чем 2). Суммарно обычное дерево захватило бы `30 + 15 = 45`.

**В симметричном дереве** нужно выбрать **одно** правило для обоих узлов сразу — CatBoost выбирает то, что максимизирует **суммарный** Gain по всем узлам уровня: `α` даёт `35`, `β` даёт `32` -> выбирается `α`, суммарный захваченный Gain — **35**.

**Разница `45 - 35 = 10`** — это буквально **цена регуляризации**, которую платит симметричное дерево за отказ от локальной гибкости. Это и есть прямой, числовой ответ на первый чек-поинт вопрос модуля: **симметричное дерево регуляризует модель сильнее, потому что оно структурно лишено степеней свободы, доступных обычному дереву** — оно физически не может подстроиться под специфику каждого отдельного узла, вынужденно принимая компромиссное, усреднённое по всем узлам уровня решение. Меньше эффективных степеней свободы -> меньше способность подогнаться под шум конкретной обучающей выборки -> ниже variance (в точности та же логика bias-variance, которую мы применяли к глубине дерева в Модуле 1 и к `num_leaves` в Модуле 7, только здесь ограничение накладывается на саму **архитектуру** дерева, а не на количество листьев или глубину).

### 2.3. Побочный эффект: сверхбыстрый инференс

Так как **все** объекты на одном уровне проверяются по **одному и тому же** правилу, путь от корня до листа для **любого** объекта — это просто последовательность из `d` бинарных сравнений (по одному на уровень), результат которых можно закодировать как `d`-битное число: бит `i` равен `1`, если объект пошёл «вправо» на уровне `i`, и `0` — если «влево». Само число, полученное такой конкатенацией битов, **и есть индекс листа** в заранее известном массиве значений — дальше не нужен обход дерева указателями (с непредсказуемыми для процессора условными переходами, каждый из которых стоит времени из-за возможного промаха предсказания ветвления, branch misprediction), достаточно вычислить число и сделать один прямой доступ по индексу в массив. Это делает предсказание симметричного дерева исключительно дешёвым и легко векторизуемым (эффективно распараллеливаемым на уровне отдельных инструкций процессора или GPU) — прямое объяснение того, почему CatBoost особенно хорошо подходит там, где критична **низкая задержка инференса** в проде (ровно то, что вы закладывали как требование `<50 мс` в спецификации FraudGuard).

## 3. Ordered Boosting — решаем скрытую проблему утечки

### 3.1. В чём тонкая проблема обычного градиентного бустинга

Вернёмся к общей схеме из Модуля 3–4: на итерации `m` мы считаем псевдо-остаток объекта `i` как `-∂l(y_i,F)/∂F`, подставляя `F = F_{m-1}(x_i)` — то есть модель, накопленную **всеми предыдущими итерациями**. Но `F_{m-1}` сама была обучена **с использованием** объекта `i` (и его истинной метки `y_i`) на всех предыдущих шагах! Получается, что при вычислении «направления коррекции» для объекта `i` мы используем модель, которая **уже видела** `y_i` — это тонкая, но реальная форма **утечки целевой переменной**, только не в данных на входе, а в самом процессе обучения.

На практике это накопленное смещение не всегда критично губительно (обычный GBM/LightGBM успешно работают на практике), но становится **особенно острым** именно при работе с категориальными признаками (раздел 4) — там эффект накопления утечки через многократное переиспользование одних и тех же данных оказывается значительно заметнее.

**Прямая параллель с уже известным вам материалом:** это концептуально тот же принцип, который вы проходили в Pandas/sklearn как **Data Leakage** (Недели 4–5, и повторно применительно к SMOTE в Модуле 12 нашего курса) — модель неявно «подглядывает» в информацию, которую не должна была бы видеть на момент предсказания.

### 3.2. Идея решения — тот же принцип, что и OOB, но встроенный в само обучение

Вспомните Модуль 2, раздел 5 — **Out-of-Bag оценку**: мы получали честную оценку качества для объекта, используя только те деревья, которые **не видели** этот объект при обучении. Ordered Boosting расширяет **тот же самый принцип** с этапа *валидации* на этап **самого обучения**: при вычислении градиента для объекта `i` мы должны использовать модель, которая **не обучалась** на объекте `i`.

### 3.3. Алгоритм

1. Сгенерировать **случайную перестановку** обучающих объектов `σ` — случайный порядок `1, ..., N`.
2. Для вычисления псевдо-остатка объекта, стоящего на позиции `k` в перестановке `σ`, использовать модель, обученную **только на объектах, стоящих в перестановке раньше него** (позиции `1, ..., k-1`).
3. На практике поддержание `N` разных «частичных» моделей (по одной на каждую возможную длину префикса) было бы прямолинейно, но крайне затратно по памяти и времени; CatBoost использует инженерный трюк — поддерживает не `N`, а логарифмически растущее число «представительных» моделей на группы длин префиксов, эффективно переиспользуя вычисления между ними (детали этой оптимизации выходят за рамки данного курса, но идея принципа — «оценка для объекта строится моделью, не видевшей его» — остаётся неизменной).

### 3.4. Численная иллюстрация идеи (упрощённо)

Пусть `N=5`, случайная перестановка задаёт порядок обработки: `[3, 1, 4, 2, 5]` (сначала объект 3, затем объект 1, затем 4, затем 2, затем 5).

Для объекта, стоящего **третьим** в этом порядке (объект **4**), его псевдо-остаток вычисляется моделью, обученной **только** на объектах `{3, 1}` — тех, что стоят раньше него в перестановке. Объекты **2** и **5** (стоящие позже в перестановке) **и сам объект 4** — не участвуют в обучении той модели, которая считает его градиент.

**Сравните с ванильным бустингом (Модули 3–6):** там градиент объекта `4` считался бы через `F_{m-1}`, которая на **предыдущих итерациях** уже обучалась на **всех пяти** объектах, включая сам объект `4` — именно эту разницу и устраняет Ordered Boosting.

**Дополнительный практический нюанс:** чтобы результат не зависел от «удачности» какой-то одной конкретной случайной перестановки (которая сама по себе вносит дисперсию — какой объект окажется «в начале», а какой «в конце», влияет на качество оценки для каждого конкретного объекта), CatBoost на практике использует **несколько** независимых случайных перестановок одновременно и усредняет результаты — это дополнительно стабилизирует итоговые оценки, снижая случайный разброс от выбора одной конкретной перестановки.

## 4. Категориальные признаки «из коробки»: Ordered Target Statistics

### 4.1. Напоминание: наивный Target Encoding и его проблема

Классический приём — заменить категорию средним значением таргета по этой категории (`mean(y | category=c)`), посчитанным по **всей** обучающей выборке. Проблема — прямая утечка: если среднее по категории `c` считается **включая** саму текущую строку, значение `y_i` этой строки напрямую участвует в формировании её же собственного признака. Даже если исключить саму строку (leave-one-out кодирование), при **многократном** переиспользовании одной и той же статистики на **множестве** итераций бустинга накопленная утечка всё равно оказывается статистически значимой — модель постепенно «запоминает» целевую переменную через категориальный признак сильнее, чем это оправдано реальной закономерностью.

### 4.2. Решение CatBoost: тот же порядок `σ`, что и для Ordered Boosting

Для объекта `i`, стоящего на позиции `k` в **той же самой** случайной перестановке `σ`, что использовалась в разделе 3, его категориальный признак кодируется статистикой, посчитанной **только** по объектам, стоящим **раньше** `i` в `σ` **и** имеющим **то же значение категории**:

In [ ]:
TS(x_i, категория=c) = ( Σ(j: σ(j)<σ(i), cat_j=c) y_j  +  a·P ) / ( count(j: σ(j)<σ(i), cat_j=c) + a )

где `P` — сглаживающее априорное значение (обычно глобальное среднее `y` по всей выборке), `a` — «вес» этого априорного значения (сколько «виртуальных» наблюдений ему приписывается). Сглаживание принципиально важно: для **первого** появления категории в порядке `σ` реальных предыдущих наблюдений просто нет (`count=0`) — без сглаживания формула была бы неопределена (деление на ноль); с ним объект получает **нейтральную**, не зависящую от собственного `y_i`, оценку `P`.

### 4.3. Численный пример «от руки»

Данные (категориальный признак `city`, таргет `y`):

| Индекс | city | y |
|---|---|---|
| 1 | A | 1 |
| 2 | B | 0 |
| 3 | A | 1 |
| 4 | C | 0 |
| 5 | B | 1 |
| 6 | A | 0 |

Случайная перестановка `σ` задаёт порядок обработки: `3, 1, 5, 2, 6, 4`. Сглаживание: `P=0.5` (нейтральный априор), `a=1`.

| Шаг | Обрабатываемый объект | city | Предыдущие объекты той же категории (в порядке σ) | sum_y / count | TS |
|---|---|---|---|---|---|
| 1 | объект 3 | A | нет (первое появление A) | 0/0 | `(0+1·0.5)/(0+1) = 0.500` |
| 2 | объект 1 | A | объект 3 (y=1) | 1/1 | `(1+0.5)/(1+1) = 0.750` |
| 3 | объект 5 | B | нет (первое появление B) | 0/0 | `(0+0.5)/(0+1) = 0.500` |
| 4 | объект 2 | B | объект 5 (y=1) | 1/1 | `(1+0.5)/(1+1) = 0.750` |
| 5 | объект 6 | A | объекты 3, 1 (y=1, y=1) | 2/2 | `(2+0.5)/(2+1) ≈ 0.833` |
| 6 | объект 4 | C | нет (первое появление C) | 0/0 | `(0+0.5)/(0+1) = 0.500` |

**Ключевое наблюдение, доказывающее отсутствие утечки:** объект **3** (первое появление категории `A` в порядке `σ`) получил кодировку **0.500** — нейтральный априор, **никак не зависящий** от его собственного `y_3=1`. Сравните с наивным Target Encoding по **всей** выборке (без учёта порядка): среднее `y` по всем строкам с `city=A` — `(1+1+0)/3 ≈ 0.667` — и это значение содержит вклад **того же самого** `y_3=1`, которое затем используется как признак **для этой же строки 3** — прямая утечка. В подходе CatBoost этого не происходит: значение признака объекта всегда вычислено **исключительно** по информации, доступной до него в случайном порядке, никогда не включая его собственную метку.

**Итог:** Ordered Target Statistics и Ordered Boosting (раздел 3) — это, по сути, **одна и та же идея**, применённая дважды в рамках одного и того же случайного порядка `σ`: и вычисление кодировки категориального признака, и вычисление псевдо-остатка для бустинга одинаково защищены принципом «никогда не используй информацию о собственной метке объекта при построении его собственного признака или направления коррекции».

## 5. Ключевые гиперпараметры CatBoost

| Параметр | Физический смысл | Связь с уже пройденным |
|---|---|---|
| `iterations` | Число деревьев (итераций бустинга) | То же, что `n_estimators` в LightGBM/sklearn (Модуль 5) |
| `depth` | Глубина **каждого** симметричного дерева | В отличие от LightGBM, здесь глубина — прямая и честная мера сложности (так как дерево всегда полностью сбалансировано, `2^depth` листьев гарантированно) |
| `l2_leaf_reg` | L2-регуляризация весов листьев | Прямой аналог `λ` из Модуля 6 / `lambda_l2` из Модуля 7 |
| `learning_rate` | Размер шага аддитивного обновления | Модуль 5 |
| `cat_features` | Список индексов/имён категориальных колонок | Указывает CatBoost, к каким столбцам применять Ordered Target Statistics (раздел 4) вместо обычной числовой обработки |
| `border_count` | Число корзин для биннинга числовых признаков (гистограммный поиск) | Аналог настройки гистограммы из Модуля 6–7 (там часто фиксированное значение ~255) |
| `one_hot_max_size` | Порог кардинальности, ниже которого категория кодируется обычным one-hot вместо Ordered TS | Экономия: для признаков с малым числом уникальных значений сложный механизм Ordered TS не даёт заметного выигрыша против простого one-hot |
| `boosting_type` | `Ordered` (по умолчанию для средних датасетов, полная защита от утечки раздела 3) или `Plain` (обычный, более быстрый бустинг без Ordered-защиты) | Явный, осознанный компромисс между скоростью и устойчивостью к переобучению — на очень больших датасетах абсолютный размер утечки относительно меньше, и `Plain` часто выбирается ради скорости |

## 6. Практика: код

### 6.1. CatBoost с сырыми категориальными признаками против LightGBM с ручным кодированием

In [1]:
import time
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from catboost import CatBoostClassifier
import lightgbm as lgb

In [3]:
# Имитация небольшого среза данных, похожего на IEEE-CIS / FraudGuard
n = 20000
rng = np.random.default_rng(42)
df = pd.DataFrame({
    "amt_log": rng.normal(4, 1.5, n),
    "hour": rng.integers(0, 24, n),
    "card4": rng.choice(["visa", "mastercard", "mir", "amex"], n, p=[0.5, 0.3, 0.15, 0.05]),
    "card6": rng.choice(["debit", "credit"], n, p=[0.7, 0.3]),
    "ProductCD": rng.choice(["W", "C", "H", "S", "R"], n),
})
# Синтетическая целевая переменная с зависимостью от категорий (для наглядности)
risk = (df["card4"] == "mir").astype(float) * 0.3 + (df["ProductCD"] == "H").astype(float) * 0.2
df["is_fraud"] = (rng.random(n) < (0.02 + risk)).astype(int)
df.head()

,amt_log,hour,card4,card6,ProductCD,is_fraud
0,4.457076,8,mir,credit,H,0
1,2.440024,8,amex,debit,W,0
2,5.125677,0,visa,debit,W,0
3,5.410847,22,visa,credit,W,0
4,1.073447,20,visa,debit,W,0


In [4]:
X = df.drop(columns=["is_fraud"])
y = df["is_fraud"]
cat_cols = ["card4", "card6", "ProductCD"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [8]:
# --- CatBoost: сырые категории, без ручного кодирования ---
start = time.perf_counter()
cb_model = CatBoostClassifier(
    iterations=300, depth=6, learning_rate=0.05,
    cat_features=cat_cols, verbose=False, random_state=42
)
cb_model.fit(X_train, y_train)
cb_time = time.perf_counter() - start
cb_pr_auc = average_precision_score(y_test, cb_model.predict_proba(X_test)[:, 1])

In [9]:
# --- LightGBM: те же категории, но закодированные вручную через category dtype ---
X_train_lgb = X_train.copy()
X_test_lgb = X_test.copy()
for col in cat_cols:
    X_train_lgb[col] = X_train_lgb[col].astype("category")
    X_test_lgb[col] = X_test_lgb[col].astype("category")

start = time.perf_counter()
lgb_model = lgb.LGBMClassifier(
    n_estimators=300, num_leaves=63, learning_rate=0.05,
    random_state=42, verbose=-1
)
lgb_model.fit(X_train_lgb, y_train, categorical_feature=cat_cols)
lgb_time = time.perf_counter() - start
lgb_pr_auc = average_precision_score(y_test, lgb_model.predict_proba(X_test_lgb)[:, 1])

In [10]:
print(f"CatBoost  | время={cb_time:.2f}с | PR-AUC={cb_pr_auc:.4f}")
print(f"LightGBM  | время={lgb_time:.2f}с | PR-AUC={lgb_pr_auc:.4f}")

CatBoost  | время=15.07с | PR-AUC=0.3578
LightGBM  | время=0.36с | PR-AUC=0.3147


**Что важно заметить:** LightGBM тоже умеет частично работать с категориями «из коробки» (через `categorical_feature` и тип `category`, раздел упоминался в Модуле 7), но использует для этого более простую внутреннюю стратегию (не Ordered TS, а упрощённый вариант группировки по частоте/градиенту), тогда как CatBoost применяет полноценный статистически обоснованный Ordered TS механизм раздела 4 — разница в качестве становится заметнее по мере роста кардинальности категориальных признаков и силы их связи с целевой переменной.

### 6.2. Иллюстрация индексации листа симметричного дерева

In [11]:
import numpy as np

In [12]:
def oblivious_leaf_index(object_features: dict, level_rules: list) -> int:
    """
    level_rules: список из d правил вида (имя_признака, порог) - по одному на уровень.
    Возвращает индекс листа как d-битное число.
    """
    leaf_index = 0
    for feature_name, threshold in level_rules:
        bit = 1 if object_features[feature_name] > threshold else 0
        leaf_index = (leaf_index << 1) | bit
    return leaf_index

In [13]:
rules = [("amt_log", 4.0), ("hour", 12)]  # depth=2 -> 4 листа

obj1 = {"amt_log": 5.2, "hour": 20}  # amt_log>4 -> 1, hour>12 -> 1 => 0b11 = 3
obj2 = {"amt_log": 2.1, "hour": 5}   # amt_log>4 -> 0, hour>12 -> 0 => 0b00 = 0

In [14]:
print("Индекс листа для obj1:", oblivious_leaf_index(obj1, rules))
print("Индекс листа для obj2:", oblivious_leaf_index(obj2, rules))

Индекс листа для obj1: 3
Индекс листа для obj2: 0


Это, конечно, упрощённая иллюстрация (реальный CatBoost работает на низкоуровневых, оптимизированных структурах), но она честно передаёт суть: индекс листа вычисляется **напрямую**, без единого условного перехода по указателям дерева — именно за счёт этого достигается высокая скорость инференса, о которой шла речь в разделе 2.3.

## 7. Частые вопросы на собеседовании

| Вопрос | На что обратить внимание в ответе |
|---|---|
| Почему симметричное дерево регуляризует модель сильнее? | Одно и то же правило разбиения для всех узлов уровня — структурное ограничение степеней свободы; дерево не может идеально подстроиться под каждый узел индивидуально, что снижает variance (аналогично общей bias-variance логике курса) |
| Как Ordered Boosting связан с Data Leakage, который вы проходили раньше? | Стандартный бустинг вычисляет градиент объекта, используя модель, уже обученную на этом же объекте в предыдущих итерациях — скрытая форма утечки. Ordered Boosting гарантирует, что градиент для объекта считается моделью, «не видевшей» его — тот же принцип, что в fit/transform-изоляции (Недели 4–5) и в OOB-оценке (Модуль 2), только применённый к самому процессу обучения бустинга |
| В чём разница между обычным Target Encoding и Ordered Target Statistics? | Обычный TE считает статистику по всей выборке (включая, возможно, саму строку) — риск утечки. Ordered TS считает статистику только по объектам, идущим раньше данного в случайном порядке — исключает прямую или косвенную утечку метки объекта в его собственный признак |
| Почему CatBoost использует несколько случайных перестановок, а не одну? | Одна конкретная перестановка сама по себе вносит случайный разброс (какие объекты окажутся «в начале», влияет на качество оценок для них) — усреднение по нескольким перестановкам стабилизирует итоговые оценки |
| Когда предпочтительнее `boosting_type='Plain'` вместо `'Ordered'`? | На очень больших датасетах, где относительная величина утечки от повторного использования данных меньше, а вычислительные затраты Ordered-режима существеннее — явный компромисс между скоростью и устойчивостью к переобучению |

## 8. Чек-поинт — попробуйте ответить без подсказок

1. Почему симметричное дерево регуляризует модель сильнее, чем обычное?
2. Как Ordered Boosting связан с проблемой Data Leakage, которую вы уже проходили в Pandas/sklearn?
3. Почему первое появление категории в случайном порядке `σ` кодируется нейтральным априорным значением, а не собственным `y` этой строки?
4. Как связаны между собой Ordered Boosting (раздел 3) и Out-of-Bag оценка (Модуль 2) — в чём общий принцип, и чем конкретно применение этого принципа отличается в двух случаях?
5. Почему симметричное дерево можно эффективно представить как `d`-битное число вместо обхода структуры указателей?

## Ответы для самопроверки

<details>
<summary>Раскрыть после того, как попробуете ответить сами</summary>

1. Потому что все узлы одного уровня обязаны использовать одно и то же правило разбиения — дерево физически лишено возможности подобрать оптимальное для каждого конкретного узла решение (в численном примере раздела 2.2 обычное дерево захватило бы Gain `45`, а симметричное — только `35`). Меньше эффективных степеней свободы структуры означает меньшую способность модели подстроиться под шум конкретной обучающей выборки — то есть более низкую variance, ту же логику bias-variance, что мы применяли к глубине дерева и числу листьев в предыдущих модулях, только теперь применённую к самой архитектуре дерева.

2. В обычном градиентном бустинге псевдо-остаток объекта `i` на итерации `m` вычисляется через модель `F_{m-1}`, которая на предыдущих итерациях уже обучалась **с использованием** объекта `i` и его истинной метки — модель «подглядывает» в информацию, которую не должна была бы знать на момент предсказания для этого объекта, ровно та же логика, что в классической утечке (например, при вычислении статистик масштабирования на полном датасете до разбиения на train/test). Ordered Boosting устраняет это, вычисляя градиент каждого объекта моделью, обученной **только** на объектах, идущих раньше него в специально сгенерированном случайном порядке — то есть моделью, которая гарантированно не видела ни сам объект, ни его метку.

3. Потому что до первого появления категории в случайном порядке `σ` **не существует** ни одного «легального» (не включающего саму эту строку) наблюдения этой категории, по которому можно было бы честно посчитать статистику — знаменатель формулы (`count`) равен нулю. Использование собственного `y` этой строки для вычисления её же признака было бы прямой утечкой; вместо этого формула подставляет нейтральный, заранее заданный априор `P` (обычно — глобальное среднее таргета), который никак не зависит от конкретной строки.

4. Общий принцип — «оценка для объекта должна строиться на основе модели/статистики, которая не использовала этот объект при своём построении» — в Модуле 2 этот принцип применяется **только для оценки качества** (OOB-ошибка): мы используем деревья, не видевшие объект, чтобы честно оценить, насколько хорошо ансамбль работает на новых данных, но сама финальная модель всё равно обучена на всех объектах через bootstrap. В Ordered Boosting тот же принцип встроен **прямо в процесс обучения**: он используется не для последующей оценки готовой модели, а для вычисления самого сигнала (градиента), на основе которого строится каждое новое дерево — то есть принцип «не подсматривать» защищает не итоговую метрику качества, а сам процесс формирования модели с самого начала.

5. Потому что в симметричном дереве путь от корня к листу для **любого** объекта состоит из одной и той же последовательности `d` бинарных проверок (одно правило на уровень) — результат каждой проверки («влево»/«вправо») можно закодировать одним битом. Последовательность этих `d` битов однозначно определяет, в каком из `2^d` листьев окажется объект, и эту последовательность битов можно напрямую интерпретировать как число — индекс в заранее подготовленном массиве значений листьев. Это заменяет последовательный обход структуры дерева (с условными переходами по указателям, каждый из которых может стоить процессору времени из-за промаха предсказания ветвления) на прямое вычисление индекса и один доступ к памяти — существенно быстрее и легче поддаётся векторизации.

</details>